# LoRA Fine-Tuning: HFACS Precondition Extraction (Qwen3-14B)

Fine-tunes `Qwen/Qwen3-14B` with LoRA to extract the **41 binary HFACS precondition sub-factors** from ASRS narratives.
Given a narrative, the model reasons inside `<think>...</think>` then outputs a JSON with 41 keys (0 or 1).

At the end, **`balanced_700_preconditions_qwen3.csv`** contains all 700 original rows.
The 500 LoRA-training rows receive 41 generated `llm_*` columns; the other 200 retain their original HFACS columns for Random Forest testing. A `lora_split` marker preserves this exact split.

## 0. Before you start

1. **Runtime**: `Runtime > Change runtime type > A100 GPU`.
2. **Generate the LoRA dataset locally** by running:
   ```bash
   python src/llm/build_lora_dataset_qwen3.py
   ```
   This reads `finetune_qwen3_config.yaml` (task = `extract_preconditions`) and writes:
   - `data/processed/lora_extract_preconditions_qwen3/lora_train.jsonl` (500 rows)
   - `data/processed/lora_extract_preconditions_qwen3/lora_test.jsonl`  (200 rows)
3. **Upload to Google Drive** into:
   ```
   MyDrive/llm_hfcas_extract_and_classify/data/qwen3_preconditions/lora_train.jsonl
   MyDrive/llm_hfcas_extract_and_classify/data/qwen3_preconditions/lora_test.jsonl
   MyDrive/llm_hfcas_extract_and_classify/data/balanced_700.csv
   ```
4. Also upload `balanced_700.csv` — it is merged with the predictions at the end to produce the preconditions CSV.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
!pip install -q -U transformers accelerate peft trl datasets huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
DRIVE_DIR   = '/content/drive/MyDrive/llm_hfcas_extract_and_classify'
TRAIN_PATH  = f'{DRIVE_DIR}/data/qwen3_preconditions/lora_train.jsonl'
TEST_PATH   = f'{DRIVE_DIR}/data/qwen3_preconditions/lora_test.jsonl'
BASE_CSV    = f'{DRIVE_DIR}/data/balanced_700.csv'
OUTPUT_DIR  = f'{DRIVE_DIR}/outputs/qwen3-14b-extract-preconditions-lora'
OUTPUT_CSV  = f'{DRIVE_DIR}/outputs/balanced_700_preconditions_qwen3.csv'

BASE_MODEL          = 'Qwen/Qwen3-14B'
MAX_SEQ_LENGTH      = 2048
NUM_EPOCHS          = 3
LEARNING_RATE       = 2e-4
PER_DEVICE_BATCH    = 2
GRAD_ACCUM_STEPS    = 4
LORA_R              = 16
LORA_ALPHA          = 16
LORA_DROPOUT        = 0.1

## 1. Load data

In [ ]:
from datasets import load_dataset

train_ds = load_dataset('json', data_files=TRAIN_PATH, split='train')
test_ds  = load_dataset('json', data_files=TEST_PATH,  split='train')

print('Train:', len(train_ds), '  Test:', len(test_ds))
print('\nSample assistant message:')
print(train_ds[0]['messages'][-1]['content'][:400])

## 2. Load model and tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.config.use_cache = False

## 3. LoRA config

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

## 4. Format dataset with chat template

In [ ]:
def format_example(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False)}

train_ds_fmt = train_ds.map(format_example, remove_columns=train_ds.column_names)
print(train_ds_fmt[0]['text'][:800])

## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds_fmt,
    peft_config=lora_config,
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved LoRA adapter to: {OUTPUT_DIR}')

## 6. Generate LoRA preconditions for the 500 training accidents

After LoRA training finishes, generate the 41 `llm_*` preconditions for those same 500 training accidents.
The other 200 accidents are not passed through LoRA; their original HFACS columns are used later as the Random Forest test features.

In [ ]:
import json
import os
from tqdm.auto import tqdm

BATCH_SIZE = 4
CHECKPOINT_EVERY = 10
CHECKPOINT_PATH = OUTPUT_DIR + '/train_inference_checkpoint.json'

model.eval()
model.config.use_cache = True
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_batch(message_batches, max_new_tokens=1000):
    prompts = [
        tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
        )
        for messages in message_batches
    ]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True).to(model.device)
    prompt_width = inputs['input_ids'].shape[1]
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.batch_decode(out[:, prompt_width:], skip_special_tokens=True)

def extract_json(text):
    json_part = text.split('</think>')[-1].strip()
    return json.loads(json_part)

def parse_and_validate(raw, expected_keys):
    try:
        pred = extract_json(raw)
    except (json.JSONDecodeError, IndexError, TypeError):
        return {}
    if not isinstance(pred, dict) or set(pred) != set(expected_keys):
        return {}
    if any(str(pred[key]) not in ('0', '1') for key in expected_keys):
        return {}
    return {key: int(pred[key]) for key in expected_keys}

def save_checkpoint(rows):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    temp_path = CHECKPOINT_PATH + '.tmp'
    with open(temp_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f)
    os.replace(temp_path, CHECKPOINT_PATH)

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, encoding='utf-8') as f:
        results = json.load(f)
    print(f'Resuming from {len(results)} saved rows')
else:
    results = []

completed = {int(row['original_index']) for row in results}
pending_indices = [
    i for i in range(len(train_ds))
    if int(train_ds[i]['original_index']) not in completed
]
last_checkpoint_count = len(results)

with tqdm(total=len(pending_indices), desc='LoRA inference') as progress:
    for start in range(0, len(pending_indices), BATCH_SIZE):
        batch_indices = pending_indices[start:start + BATCH_SIZE]
        examples = [train_ds[i] for i in batch_indices]
        message_batches = [example['messages'][:2] for example in examples]
        raw_outputs = generate_batch(message_batches)

        batch_rows = []
        invalid_positions = []
        for position, (example, raw) in enumerate(zip(examples, raw_outputs)):
            true = extract_json(example['messages'][2]['content'])
            pred = parse_and_validate(raw, true.keys())
            batch_rows.append({'example': example, 'raw': raw, 'true': true, 'pred': pred})
            if not pred:
                invalid_positions.append(position)

        # Retry all malformed/truncated rows together instead of one at a time.
        if invalid_positions:
            retry_messages = [message_batches[position] for position in invalid_positions]
            retry_outputs = generate_batch(retry_messages, max_new_tokens=1400)
            for position, retry_raw in zip(invalid_positions, retry_outputs):
                batch_rows[position]['raw'] = retry_raw
                batch_rows[position]['pred'] = parse_and_validate(
                    retry_raw, batch_rows[position]['true'].keys()
                )

        for row in batch_rows:
            example = row['example']
            results.append({
                'original_index': int(example['original_index']),
                'split': example['split'],
                'pred': row['pred'],
                'true': row['true'],
                'raw_output': row['raw'],
            })

        progress.update(len(examples))
        if len(results) - last_checkpoint_count >= CHECKPOINT_EVERY:
            save_checkpoint(results)
            last_checkpoint_count = len(results)

results.sort(key=lambda row: int(row['original_index']))
save_checkpoint(results)
parsed_count = sum(bool(row['pred']) for row in results)
print(f'Inference done. Valid 41-field JSON: {parsed_count}/{len(results)}')
print('LoRA-feature rows:', len(results))

## 7. Diagnostic agreement on the 500 LoRA-training rows

In [ ]:
# This is training-set agreement, not held-out LoRA evaluation.
factor_keys = list(results[0]['true'].keys()) if results else []
print(f'Checking {len(factor_keys)} precondition factors on {len(results)} training examples\n')

accuracies = {}
for key in factor_keys:
    correct = sum(
        1 for r in results
        if str(r['pred'].get(key, -1)) == str(r['true'].get(key, -1))
    )
    accuracies[key] = correct / len(results)
    print(f'  {key}: {correct}/{len(results)} = {accuracies[key]:.3f}')

print(f'\nMean per-factor accuracy: {sum(accuracies.values()) / len(accuracies):.3f}')

## 8. Save preconditions CSV

Saves all 700 original accidents in one CSV. The 500 `train` rows contain generated `llm_*` features.
The 200 `test` rows keep their original HFACS feature columns; their `llm_*` cells remain empty because RF tests them using the original columns.
The `lora_split` marker ensures Random Forest uses the exact same 500/200 accidents.

In [ ]:
import pandas as pd

base_df = pd.read_csv(BASE_CSV)

# Build llm_* columns from predictions, keyed by original_index
llm_rows = []
for r in results:
    row = {'original_index': r['original_index']}
    for key in r['true']:
        val = r['pred'].get(key, 0)
        row[f'llm_{key}'] = int(val) if str(val) in ('0', '1') else 0
    llm_rows.append(row)

llm_df = pd.DataFrame(llm_rows).set_index('original_index')

# Mark all 700 rows with the exact split used to build the LoRA datasets
split_rows = [
    {'original_index': ex['original_index'], 'lora_split': ex['split']}
    for ex in list(train_ds) + list(test_ds)
]
split_df = pd.DataFrame(split_rows).set_index('original_index')

# Only the 500 train rows receive llm_* values; all original columns remain for all 700 rows
output_df = base_df.join(split_df).join(llm_df).sort_index()
assert len(output_df) == 700, f'Expected 700 rows, got {len(output_df)}'
assert output_df['lora_split'].value_counts().to_dict() == {'train': 500, 'test': 200}
assert output_df.loc[output_df['lora_split'] == 'train', llm_df.columns].notna().all().all()
assert output_df.loc[output_df['lora_split'] == 'test', llm_df.columns].isna().all().all()

output_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(output_df)} rows, {len(llm_df.columns)} llm_* columns → {OUTPUT_CSV}')
print('\nllm_* columns:')
print([c for c in output_df.columns if c.startswith('llm_')])

In [ ]:
# Also save raw predictions for debugging
json_path = OUTPUT_DIR + '/train_500_predictions.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print(f'Saved raw predictions → {json_path}')